# 02 - AI/ML Role Filtering

This notebook applies reproducible rule-based relevance scoring to identify AI/ML-oriented jobs.

## Load Normalized Dataset

In [1]:
from __future__ import annotations

import polars as pl

from job_market_copilot.config import get_settings
from job_market_copilot.processing.relevance import score_relevance

settings = get_settings()
normalized_path = settings.resolved_artifacts_dir / 'jobs_normalized.parquet'
all_df = pl.read_parquet(normalized_path)
all_df.shape

(128, 14)

## Score Relevance

In [2]:
scored_df = score_relevance(all_df, threshold=settings.ai_keyword_threshold)
ai_df = scored_df.filter(pl.col('is_ai_role_rule'))
print({'total': scored_df.height, 'ai_candidates': ai_df.height})
ai_df.select(['source','title','company','ai_relevance_score']).head(15)

{'total': 128, 'ai_candidates': 40}


source,title,company,ai_relevance_score
str,str,str,i64
"""remotive""","""Business Transformation Lead""","""Expion Health""",5
"""remotive""","""Mid/Senior AI Cinematic Video …","""EverAI""",10
"""remotive""","""Customer Operations & Writing …","""Clerky, Inc.""",5
"""remotive""","""Office Assistant""","""Coalition Technologies""",3
"""remotive""","""Staff Product Engineer (Belo H…","""LawnStarter""",5
…,…,…,…
"""remotive""","""Senior Independent AI Engineer…","""A.Team""",7
"""remotive""","""Staff Software Engineer, Produ…","""LawnStarter""",6
"""remotive""","""Staff Software Engineer, Produ…","""LawnStarter""",6


## Persist Scored Artifacts

In [3]:
scored_df.write_parquet(settings.resolved_artifacts_dir / 'jobs_scored.parquet')
ai_df.write_parquet(settings.resolved_artifacts_dir / 'jobs_ai_candidates.parquet')

## Diagnostics by Source

In [4]:
ai_df.group_by('source').len(name='count').sort('count', descending=True)

source,count
str,u32
"""wwr""",23
"""remotive""",17
